In [25]:
import sys, os, re

import qubx
from qubx import logger

%qubxd

%load_ext autoreload
%autoreload 2

from qubx.core.basics import Signal, TimestampedDict
from qubx.core.lookups import lookup
from qubx.core.interfaces import IStrategy, IStrategyContext, TriggerEvent, MarketEvent, IStrategyInitializer
from qubx.data.registry import StorageRegistry
from qubx.data.transformers import TypedGenericSeries

from qubx.backtester.simulator import simulate

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [29]:
stor = StorageRegistry.get("csv::../../tests/data/storages/csv/")
stor.get_exchanges()

['BINANCE.UM', 'KRAKEN']

In [30]:
stor.get_reader("BINANCE.UM", "SWAP").get_time_range("AAVEUSDT", "ohlc(1h)")

(numpy.datetime64('2023-06-01T00:00:00'),
 numpy.datetime64('2023-08-01T00:00:00'))

In [119]:
stor.get_reader("BINANCE.UM", "SWAP").get_time_range("BTCUSDT", "quote")

(numpy.datetime64('2017-08-24T13:01:12'),
 numpy.datetime64('2017-08-24T13:01:59'))

In [120]:
stor.get_reader("KRAKEN", "SWAP").get_time_range("AAVEUSD", "ohlc(1h)")

(numpy.datetime64('2023-06-01T00:00:00'),
 numpy.datetime64('2023-08-01T00:00:00'))

In [13]:
stor.get_reader("KRAKEN", "SWAP").read("BTCUSD", "quote").to_records()[:3]

[[2025-12-01T00:00:00.000000000]	90388.00000 (0.01) | 90389.00000 (0.01),
 [2025-12-01T00:00:01.000000000]	90380.00000 (0.01) | 90386.00000 (0.01),
 [2025-12-01T00:00:02.000000000]	90364.00000 (0.01) | 90370.00000 (0.01)]

In [17]:
stor.get_reader("KRAKEN", "SWAP").read("BTCUSD", "quote").transform(TypedGenericSeries())

                         bid      ask  bid_size  ask_size
time                                                     
2025-12-01 00:00:00  90388.0  90389.0    0.0101    0.0113
2025-12-01 00:00:01  90380.0  90386.0    0.0101    0.0113
2025-12-01 00:00:02  90364.0  90370.0    0.0109    0.0113
2025-12-01 00:00:03  90366.0  90367.0    0.0050    0.0060
2025-12-01 00:00:04  90358.0  90361.0    0.0005    0.0236
...                      ...      ...       ...       ...
2025-12-01 23:59:55  86317.0  86318.0    0.0022    0.0071
2025-12-01 23:59:56  86317.0  86318.0    0.0022    0.0131
2025-12-01 23:59:57  86317.0  86318.0    0.0022    0.1869
2025-12-01 23:59:58  86291.0  86298.0    0.0641    0.0391
2025-12-01 23:59:59  86290.0  86291.0    0.0050    0.1792

[83631 rows x 4 columns]

In [ ]:
lookup.find_instruments("HYPERLIQUID")

In [19]:
class Test1(IStrategy):
    def on_init(self, initializer: IStrategyInitializer):
        pass

    def on_market_data(self, ctx: IStrategyContext, data: MarketEvent) -> list[Signal] | Signal | None:
        pass

In [24]:
simulate(
    Test1(), 
    {
        "ohlc": stor, "quote": stor,
    }, 
    capital=1000,
    start="2020-01-01",
    instruments=[
        "BINANCE.UM:SWAP:BTCUSDT",
        "KRAKEN:SWAP:BTCUSD"
    ]
)

2026-02-04 15:37:10.987 [ ⚠️ ] (utils) qubx.backtester.utils:_process_single_symbol_or_instrument:278 - Can't find instrument for specified symbol (KRAKEN:SWAP:BTCUSD) - ignoring !


SimulationConfigError: Unsupported data provider type: <class 'qubx.data.storages.csv.CsvStorage'>

In [33]:
xs = StorageRegistry.get("qdb::quantlab")

In [34]:
rs = xs.get_reader("HYPERLIQUID", "SWAP")
rb = xs.get_reader("BINANCE.UM", "SWAP")

In [36]:
rs.get_data_types("BTCUSDC")

[funding_payment,
 orderbook,
 'ohlc(1min)',
 'ohlc(2min)',
 'ohlc(3min)',
 'ohlc(5min)',
 'ohlc(10min)',
 'ohlc(15min)',
 'ohlc(30min)',
 'ohlc(1h)',
 'ohlc(2h)',
 'ohlc(3h)',
 'ohlc(4h)',
 'ohlc(6h)',
 'ohlc(8h)',
 'ohlc(12h)',
 'ohlc(1d)',
 'ohlc(1w)',
 open_interest,
 quote,
 funding_rate]

In [48]:
for x in rs.read(["BTCUSDC", "AAVEUSDC", "ETHUSDC", "LTCUSDC"], "ohlc(1h)", "2025-01-01", "2026-01-01 00:10"):
    # x.to_pd().to_csv(f"../../tests/data/storages/csv/HYPERLIQUID/SWAP/{x.data_id}.1H.csv.gz")
    print(x)

LTCUSDC(ohlc(1h))[2025-01-01 00:00:00 : 2026-01-01 00:00:00] : (8761 x 11)
ETHUSDC(ohlc(1h))[2025-01-01 00:00:00 : 2026-01-01 00:00:00] : (8761 x 11)
BTCUSDC(ohlc(1h))[2025-01-01 00:00:00 : 2026-01-01 00:00:00] : (8761 x 11)
AAVEUSDC(ohlc(1h))[2025-01-01 00:00:00 : 2026-01-01 00:00:00] : (8761 x 11)


In [53]:
xx1 = rs.read(["BTCUSDC", "AAVEUSDC", "ETHUSDC", "LTCUSDC"], "ohlc(1h)", "2025-01-01", "2026-01-01 00:10")

In [ ]:
xx1.to_pd(1).head()